# 4. Trying to break the answer

**Goal:** run the checks of Section 4.6 on the Q1 estimate. Code makes every pass/fail decision with
thresholds from `config.yaml`; nothing here is judged by eye.

To keep this notebook quick, the half-sample check uses 10 subsets instead of 20. The full-strength
version is `python -m causallab refute`.

## Setup

Imports and configuration. Every threshold and setting comes from `config.yaml`. Set `SYNTHETIC = True` to run on the synthetic file built by `python -m causallab prepare --synthetic`.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json
import numpy as np
import pandas as pd
from causallab.config import load_config, load_spec, results_dir

cfg = load_config()
SYNTHETIC = False   # True = run on the synthetic Expedia-like file (no Kaggle data needed)
pd.set_option("display.width", 140)
cfg["refuters"]["subset_n"] = 10   # notebook speed only; the CLI uses the config value (20)

## Estimate first

**What:** the five estimators on the reference specification, as in notebook 03.

**Expect:** a few minutes.

In [2]:
from causallab import data, estimators as E, features, refuters as R
spec = load_spec(cfg); cov = spec["adjustment_set"]
df = features.q1_rows(data.load_arm("observational", cfg, SYNTHETIC))
est = E.estimate_all(df, "top3", "click_bool", cov, "srch_id", cfg)
primary = est["estimates"]["aipw"]
kept = df[est["keep"]].reset_index(drop=True)
t, y, e = kept.top3.to_numpy(), kept.click_bool.to_numpy(float), est["propensity"][est["keep"]]
print(f"AIPW {primary.estimate*100:.2f} pp")

AIPW 6.61 pp


## Overlap

**What:** the distribution of propensity scores, and the share of rows outside [0.01, 0.99].

**Why:** if some hotels are almost certain to be (or not to be) on top, there is no comparable
control for them, and weights explode.

**Expect:** no rows trimmed: the ranker is far from deterministic given the covariates.

In [3]:
R.overlap(est, cfg)

{'check': 'overlap',
 'trimmed_share': 0.0,
 'trim': [0.01, 0.99],
 'propensity_quantiles': {0.01: 0.11457810695221113,
  0.05: 0.1422455232464433,
  0.5: 0.31485292242845153,
  0.95: 0.671543354558319,
  0.99: 0.8342368327120256},
 'rule': 'share of rows outside [0.01, 0.99] <= 0.3',
 'passed': True}

## Covariate balance after weighting

**What:** SMDs of every adjustment variable between top-3 and control, after weighting with trimmed,
stabilised inverse propensity weights.

**Why:** if weighting worked, the weighted groups look alike on everything we adjusted for.

**Expect:** all below 0.1 (large before weighting, tiny after).

In [4]:
bal = R.balance(kept, cov, t, e, cfg)
print("passed:", bal["passed"], "| worst:", bal["worst_covariate"], round(bal["max_abs_smd_after"], 3))
pd.DataFrame({"before": bal["smd_before"], "after": bal["smd_after"]}).sort_values("before", key=abs, ascending=False).head(10).round(3)

passed: True | worst: srch_query_affinity_score 0.024


,before,after
promotion_flag,0.261,0.018
loc2_missing,-0.241,-0.001
loc2_rel,0.240,0.017
prop_location_score2,0.234,0.014
star_rel,0.227,0.019
prop_starrating,0.151,-0.004
review_rel,0.107,0.014
prop_log_historical_price,0.107,0.014
star_zero,-0.100,-0.015
prop_review_score,0.090,0.006


## Placebo treatment

**What:** permute top-3 across the whole sample and re-estimate, five times, and average the five estimates.

**Why:** a permuted treatment is unrelated to anything, so any 'effect' points to a bug. (The spec
shuffled within each search; that leaks the true effect when searches differ in their number of
treated rows, so the permutation is across the whole sample.)

**Rule:** the average must lie within ±1.96 *single-run* standard errors of zero. Averaging removes most of one permutation's chance noise (with one permutation, the check fired by chance on 3 of 53 runs). This over-corrects: the average of five is held to about 4.4 of its own standard errors (1.96 × √5), so chance false alarms almost never happen and only a systematic placebo effect bigger than about two single-run standard errors is caught. It is not a 5% test. Tested against its own standard error, the Q1 placebo still passes easily (z = −0.06); the Q3 clicks placebo does not (z = −2.4, see the README).

**Expect:** an interval that includes zero.

In [5]:
R.placebo(df, "top3", "click_bool", cov, "srch_id", cfg)

{'check': 'placebo',
 'estimate': -1.432480275398813e-05,
 'ci': [-0.0010752179930143537, 0.0010465683875063776],
 'estimates': [-0.0001092998979141927,
  -0.000200733827395741,
  -0.0008778849660832105,
  0.0006215181514799805,
  0.000494776526143223],
 'single_run_se': 0.0005412819820305658,
 'rule': 'average of 5 permutation estimates within +/-1.96 single-run standard errors of zero',
 'passed': True}

## Random common cause and subset stability

**What:** add a pure-noise covariate and re-estimate; then re-estimate on random half-samples of
searches.

**Why:** a stable estimate should not move when noise is added, and half-samples should mostly land
inside the full-sample interval.

**Expect:** a shift well under half a standard error; at least 80% of half-samples inside (in practice nearly all).

In [6]:
print(R.random_common_cause(df, "top3", "click_bool", cov, "srch_id", cfg, primary))
sub = R.subset_stability(df, "top3", "click_bool", cov, "srch_id", cfg, primary)
print({k: v for k, v in sub.items() if k != "estimates"})

{'check': 'random_common_cause', 'estimate': 0.06608228560856211, 'shift_in_se': 0.019570097671172782, 'rule': 'estimate moves by < 0.5 standard errors', 'passed': True}


{'check': 'subset_stability', 'share_inside': 1.0, 'rule': 'at least 80% of 10 half-sample estimates inside the full-sample 95% interval', 'passed': True}


## Held-out covariate (reported, not pass or fail)

**What:** leave location score 2 out, together with the two columns computed from it (its missing flag
and its value relative to the search), and estimate the 'effect' of top-3 on it.

**Why:** in the observational arm the ranker uses hotel attributes, so top-3 hotels *should* differ
on them; this measures how much. It is not a pass/fail check.

**Expect:** a clearly nonzero value.

In [7]:
R.held_out_covariate(kept, t, cov, "prop_location_score2", kept.srch_id.to_numpy())

{'check': 'held_out_covariate',
 'covariate': 'prop_location_score2',
 'also_left_out': ['loc2_missing', 'loc2_rel'],
 'estimate': 0.04347110528351478,
 'estimate_in_sd': 0.25465743934302176,
 'ci': [0.04283658093510249, 0.04410562963192707],
 'rule': 'reported, not pass or fail',
 'passed': None}

## Sensitivity to a hidden confounder

**What:** omitted-variable bias bounds (Cinelli & Hazlett, 2020) on the linear regression: if a hidden
confounder were as strong as each observed covariate, how far would the estimate move? Also the
robustness value and the E-value.

**Why:** no check above can detect a confounder that is not in the data, such as the ranker's own
score. This says how strong one would need to be to explain the effect away.

**Expect:** the effect keeps its sign even with a confounder as strong as the strongest covariate.

In [8]:
sens = R.sensitivity(y, t, kept[cov], primary, cfg, groups=kept.srch_id.to_numpy())
print({k: sens[k] for k in ("passed", "strongest_benchmark_covariate", "robustness_value_q1", "e_value")})
sens["bound_at_strongest"]

{'passed': True, 'strongest_benchmark_covariate': 'n_results', 'robustness_value_q1': 0.12103046856906508, 'e_value': 3.3869439129927725}


{'r2dz_x': 0.006953321758314387,
 'r2yz_dx': 0.004573844860100951,
 'bias': 0.0035261995630224514,
 'adjusted_estimate': 0.07691205623567982}

## The uncomfortable lesson

Every check can pass while the estimate still misses the benchmark. The checks test what the data can
reveal; a confounder the file does not contain (the ranker's information), or a covariate that is
secretly an outcome, can leave all of them quiet.